# HW5 預訓練一個小 GPT，再載入 GPT-2 的權重

> 說明：<https://github.com/chang-ye-tu/genai/blob/main/hw/hw5.md>　取材：Raschka《Build a Large Language Model (From Scratch)》第 5 章（套件 `llms-from-scratch`）
> 授權：本筆記本呼叫並改寫 Sebastian Raschka 的開源專案 <https://github.com/rasbt/LLMs-from-scratch>（Apache-2.0，© Sebastian Raschka）的範例程式；改寫部分（本課程的講解、問題與資料）同樣以 Apache-2.0 散布，完整授權文本見 repo 的 `LICENSES/Apache-2.0.txt`。
> 做法：**執行階段 → 變更執行階段類型 → T4 GPU**，由上而下逐格執行；看到「✍️ 請回答」就把觀察寫進該文字格。全部跑完後「檔案 → 下載 → .ipynb」上傳 iLearn，再作答實作測驗。
> **請勿更改模型名稱、版本、隨機種子與資料檔**，否則實作測驗的數值題會對不上。
> 需要 T4 GPU（訓練約 2–3 分鐘）。
> 核心段落：第 0–3、5 節（實作測驗只出這些段落的題目）；第 4 節為選做。


In [ ]:
import hashlib
def sha256_of(path, expected=None):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    digest = h.hexdigest()
    if expected is not None and digest != expected:
        raise RuntimeError(f"{path} 的 SHA-256 與課程固定版本不符：{digest[:12]}… ≠ {expected[:12]}…；請刪除檔案重新下載，或到 Q&A 回報")
    print(f"SHA-256 OK：{path} ({digest[:12]}…)")
    return digest

def fetch(path, urls, expected):
    # 依序嘗試 urls：每個網址下載後立刻驗 SHA-256，不符就刪掉換下一個來源；已存在且正確的檔案直接使用；全部來源都失敗才報錯
    import os, requests
    urls = [urls] if isinstance(urls, str) else list(urls)
    def _ok():
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
        return h.hexdigest() == expected
    if os.path.exists(path):
        if _ok():
            print(f"SHA-256 OK：{path} ({expected[:12]}…)"); return path
        print(f"{path} 的 SHA-256 與課程固定版本不符，刪除後重新下載"); os.remove(path)
    for url in urls:
        try:
            with requests.get(url, timeout=600, stream=True) as r:
                r.raise_for_status()
                with open(path, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
        except Exception as e:
            print("下載失敗：", url, e)
            if os.path.exists(path): os.remove(path)
            continue
        if _ok():
            print("下載自", url); print(f"SHA-256 OK：{path} ({expected[:12]}…)"); return path
        print(f"{url} 下載的內容 SHA-256 不符，改用下一個來源"); os.remove(path)
    raise RuntimeError(f"{path} 所有來源都無法取得正確的檔案（下載失敗或 SHA-256 不符）；請到 Q&A 回報")

# @title 第 0 節：安裝與資料
%pip -q install --no-deps llms-from-scratch==1.0.19
%pip -q install tiktoken==0.14.0 safetensors==0.8.0
import torch, tiktoken, requests, os, math
from llms_from_scratch.ch02 import create_dataloader_v1
from llms_from_scratch.ch04 import GPTModel
from llms_from_scratch.ch05 import (calc_loss_loader, train_model_simple, generate, text_to_token_ids, token_ids_to_text, plot_losses)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("裝置：", device)
import platform, importlib.metadata as _meta
def _v(p):
    try: return _meta.version(p)
    except Exception: return "missing"  # metadata 查不到時印 missing；若同一格更早的 import 已失敗，程式到不了這裡，check_submissions 會判「無版本資訊／執行錯誤」
print("VERSIONS", "python=" + platform.python_version(), "torch=" + torch.__version__, *[p + "=" + _v(p) for p in ["llms-from-scratch", "tiktoken", "safetensors"]])
fetch("the-verdict.txt", "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/d1d29d055561666740d93e91a62af0c4726e103b/ch02/01_main-chapter-code/the-verdict.txt", "b41e41a68f0398a3154ae69e2e4c0e2694e17fe0d66730536837f1b01935b31f")
text = open("the-verdict.txt", encoding="utf-8").read()
tokenizer = tiktoken.get_encoding("gpt2")
print("字元數：", len(text), "| token 數：", len(tokenizer.encode(text)))
print(text[:200])


## 第 1 節 訓練資料：一篇 2 萬字元的短篇小說

預訓練的「標註」就是文字本身：輸入一段 256 個 token，目標是每個位置的下一個 token。切 90% 訓練、10% 驗證。


In [ ]:
split_idx = int(0.90 * len(text))
train_data, val_data = text[:split_idx], text[split_idx:]
torch.manual_seed(123)
train_loader = create_dataloader_v1(train_data, batch_size=2, max_length=256, stride=256, drop_last=True, shuffle=True, num_workers=0)
val_loader = create_dataloader_v1(val_data, batch_size=2, max_length=256, stride=256, drop_last=False, shuffle=False, num_workers=0)
train_tokens = sum(x.numel() for x, _ in train_loader); val_tokens = sum(x.numel() for x, _ in val_loader)
print("訓練 token：", train_tokens, "| 驗證 token：", val_tokens, "| 訓練批次數：", len(train_loader))
x, y = next(iter(train_loader))
print("輸入形狀：", tuple(x.shape), "| 目標就是輸入右移一格：", (x[0, 1:6] == y[0, :5]).all().item())


## 第 2 節 訓練前的損失：和「亂猜」比

交叉熵損失＝對正確 token 的機率取 −log。完全均勻亂猜時損失是 ln(50257)。


In [ ]:
GPT_CONFIG_124M = {"vocab_size": 50257, "context_length": 256, "emb_dim": 768, "n_heads": 12, "n_layers": 12, "drop_rate": 0.1, "qkv_bias": False}
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M).to(device)
with torch.no_grad():
    print("訓練前 訓練損失：", round(calc_loss_loader(train_loader, model, device), 3), "| 驗證損失：", round(calc_loss_loader(val_loader, model, device), 3))
print("ln(50257) =", round(math.log(50257), 3))


✍️ **請回答 2-1**：為什麼隨機初始化的模型損失接近 ln(50257)？上一格印出的初始損失（接近 ln(50257)）對應的「正確 token 機率」大約是多少（提示：e 的負「損失值」次方）？

（在這裡作答）


## 第 3 節 訓練 10 個 epoch

每 5 步印一次訓練／驗證損失，並用「Every effort moves you」接龍看進步。


In [ ]:
NUM_EPOCHS = 10
torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
train_losses, val_losses, tokens_seen = train_model_simple(model, train_loader, val_loader, optimizer, device,
    num_epochs=NUM_EPOCHS, eval_freq=5, eval_iter=5, start_context="Every effort moves you", tokenizer=tokenizer)


In [ ]:
epochs_tensor = torch.linspace(0, NUM_EPOCHS, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)
print("最後的訓練損失：", round(train_losses[-1], 3), "| 驗證損失：", round(val_losses[-1], 3))


✍️ **請回答 3-1**：訓練損失和驗證損失從第幾個 epoch 開始分開？這叫什麼現象（第 7、8 單元）？只有幾千個 token 的資料（實際 token 數見第 0 節的輸出）為什麼一定會發生？

（在這裡作答）


## 第 4 節（選做）解碼策略：temperature 與 top-k

用剛訓練好的模型，比較 greedy、temperature 1.5、top-k 5。


In [ ]:
model.eval()
idx = text_to_token_ids("Every effort moves you", tokenizer).to(device)
for name, kw in [("greedy", dict(temperature=0.0)), ("temperature 1.5", dict(temperature=1.5)), ("top-k 5, temp 1.0", dict(temperature=1.0, top_k=5))]:
    torch.manual_seed(123)
    out = generate(model=model, idx=idx, max_new_tokens=25, context_size=GPT_CONFIG_124M["context_length"], **kw)
    print(f"[{name}] {token_ids_to_text(out, tokenizer)!r}")


✍️ **請回答 4-1**：哪一種最像原文？哪一種最「有創意」但最容易出現不存在的詞？和 HW1 第 3 節在 Qwen 上的觀察一致嗎？

（在這裡作答）


## 第 5 節 載入 OpenAI 訓練好的 GPT-2 124M

同樣的層數與維度（12 層、768 維、12 頭），參數換成 OpenAI 用 40 GB 文字訓練出來的權重；設定上有兩點不同：上下文長度 1,024（我們訓練時用 256）、Q/K/V 有偏差項（`qkv_bias=True`）。看損失與生成的差別。


In [ ]:
import hashlib
def sha256_of(path, expected=None):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    digest = h.hexdigest()
    if expected is not None and digest != expected:
        raise RuntimeError(f"{path} 的 SHA-256 與課程固定版本不符：{digest[:12]}… ≠ {expected[:12]}…；請刪除檔案重新下載，或到 Q&A 回報")
    print(f"SHA-256 OK：{path} ({digest[:12]}…)")
    return digest

def fetch(path, urls, expected):
    # 依序嘗試 urls：每個網址下載後立刻驗 SHA-256，不符就刪掉換下一個來源；已存在且正確的檔案直接使用；全部來源都失敗才報錯
    import os, requests
    urls = [urls] if isinstance(urls, str) else list(urls)
    def _ok():
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
        return h.hexdigest() == expected
    if os.path.exists(path):
        if _ok():
            print(f"SHA-256 OK：{path} ({expected[:12]}…)"); return path
        print(f"{path} 的 SHA-256 與課程固定版本不符，刪除後重新下載"); os.remove(path)
    for url in urls:
        try:
            with requests.get(url, timeout=600, stream=True) as r:
                r.raise_for_status()
                with open(path, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
        except Exception as e:
            print("下載失敗：", url, e)
            if os.path.exists(path): os.remove(path)
            continue
        if _ok():
            print("下載自", url); print(f"SHA-256 OK：{path} ({expected[:12]}…)"); return path
        print(f"{url} 下載的內容 SHA-256 不符，改用下一個來源"); os.remove(path)
    raise RuntimeError(f"{path} 所有來源都無法取得正確的檔案（下載失敗或 SHA-256 不符）；請到 Q&A 回報")

# GPT-2 124M 預訓練權重：從 Hugging Face 下載 safetensors（548 MB），對應到我們自己實作的 GPTModel
import os, requests
from safetensors.torch import load_file
SF = "model-gpt2.safetensors"
fetch(SF, f"https://huggingface.co/openai-community/gpt2/resolve/607a30d783dfa663caf39e06633721c8d4cfcd7e/model.safetensors", "248dfc3911869ec493c76e65bf2fcf7f615828b0254c12b473182f0f81d3a707")  # 已存在且 hash 正確就不重下；不符會自動刪除重下一次
state_dict = load_file(SF)
print("張量數：", len(state_dict), "| 檔案大小 MB：", round(os.path.getsize(SF) / 1e6))

def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(right.detach().clone())

def load_hf_weights_into_gpt(gpt, p):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, p["wpe.weight"])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, p["wte.weight"])
    for b in range(len(gpt.trf_blocks)):
        t = gpt.trf_blocks[b]
        qw, kw, vw = torch.chunk(p[f"h.{b}.attn.c_attn.weight"], 3, dim=-1)
        qb, kb, vb = torch.chunk(p[f"h.{b}.attn.c_attn.bias"], 3, dim=-1)
        t.att.W_query.weight = assign(t.att.W_query.weight, qw.T); t.att.W_query.bias = assign(t.att.W_query.bias, qb)
        t.att.W_key.weight = assign(t.att.W_key.weight, kw.T);     t.att.W_key.bias = assign(t.att.W_key.bias, kb)
        t.att.W_value.weight = assign(t.att.W_value.weight, vw.T); t.att.W_value.bias = assign(t.att.W_value.bias, vb)
        t.att.out_proj.weight = assign(t.att.out_proj.weight, p[f"h.{b}.attn.c_proj.weight"].T)
        t.att.out_proj.bias = assign(t.att.out_proj.bias, p[f"h.{b}.attn.c_proj.bias"])
        t.ff.layers[0].weight = assign(t.ff.layers[0].weight, p[f"h.{b}.mlp.c_fc.weight"].T)
        t.ff.layers[0].bias = assign(t.ff.layers[0].bias, p[f"h.{b}.mlp.c_fc.bias"])
        t.ff.layers[2].weight = assign(t.ff.layers[2].weight, p[f"h.{b}.mlp.c_proj.weight"].T)
        t.ff.layers[2].bias = assign(t.ff.layers[2].bias, p[f"h.{b}.mlp.c_proj.bias"])
        t.norm1.scale = assign(t.norm1.scale, p[f"h.{b}.ln_1.weight"]); t.norm1.shift = assign(t.norm1.shift, p[f"h.{b}.ln_1.bias"])
        t.norm2.scale = assign(t.norm2.scale, p[f"h.{b}.ln_2.weight"]); t.norm2.shift = assign(t.norm2.shift, p[f"h.{b}.ln_2.bias"])
    gpt.final_norm.scale = assign(gpt.final_norm.scale, p["ln_f.weight"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, p["ln_f.bias"])
    gpt.out_head.weight = assign(gpt.out_head.weight, p["wte.weight"])  # 權重共享：輸出層 = 詞嵌入

GPT2_CFG = {"vocab_size": 50257, "context_length": 1024, "emb_dim": 768, "n_heads": 12, "n_layers": 12, "drop_rate": 0.0, "qkv_bias": True}  # 與原始 GPT-2 一致：有 QKV 偏差、上下文 1024

gpt2 = GPTModel(GPT2_CFG)
load_hf_weights_into_gpt(gpt2, state_dict)
gpt2.to(device).eval()
with torch.no_grad():
    print("GPT-2 124M 在《The Verdict》上的 驗證損失：", round(calc_loss_loader(val_loader, gpt2, device), 3), "| 訓練損失：", round(calc_loss_loader(train_loader, gpt2, device), 3))
for start in ["Every effort moves", "The capital of Taiwan is", "臺灣最高的山是"]:
    idx = text_to_token_ids(start, tokenizer).to(device)
    out = generate(model=gpt2, idx=idx, max_new_tokens=20, context_size=1024, temperature=0.0)
    print(f"[{start!r} → {idx.shape[1]} tokens] {token_ids_to_text(out, tokenizer)!r}")


✍️ **請回答 5-1**：我們不知道 GPT-2 的訓練資料（WebText）有沒有這篇 1908 年的公版小說，但它的驗證損失遠低於我們用這篇小說訓練出來的模型（它在驗證集上多少？）。除了「可能看過」之外，還有什麼合理的解釋？中文提示的輸出為什麼是亂碼、而且短短 7 個中文字要十幾個 token（確切數字見上一格的輸出）？這和第 9 單元講的預訓練資料有什麼關係？

（在這裡作答）


## ✍️ AI 使用聲明（必填）

| 項目 | 內容 |
|------|------|
| 使用的工具 | （例如：ChatGPT 免費版、Colab 內建 Gemini） |
| 用在哪些工作 | （例如：解釋錯誤訊息、幫我看懂某一格程式） |
| 我自己完成的部分 | （例如：全部執行、所有 ✍️ 回答） |
| 我如何驗證 AI 的說法 | （例如：實際執行、對照投影片） |

姓名／學號：
